In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datetime import datetime
import shii
import pandas as pd
from dotenv import load_dotenv
import geopandas as gpd
from shapely import Point

In [ ]:
heat_incidents = shii.download_heat_incidents(
    app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
    limit=10000,
    start_timestamp="2024-05-01",
    end_timestamp="2024-10-31"
)
hydrant_complaints = shii.download_311_complaints(
    app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
    complaint_type="hydrant",
    limit=10000,
    start_timestamp="2024-05-01",
    end_timestamp="2024-10-31"
)
ac_complaints = shii.download_311_complaints(
    app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
    complaint_type="ac",
    limit=10000,
    start_timestamp="2024-05-01",
    end_timestamp="2024-10-31"
)
power_complaints = shii.download_311_complaints(
    app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
    complaint_type="power",
    limit=10000,
    start_timestamp="2024-05-01",
    end_timestamp="2024-10-31"
)
ventilation_complaints = shii.download_311_complaints(
    app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
    complaint_type="ventilation",
    limit=10000,
    start_timestamp="2024-05-01",
    end_timestamp="2024-10-31"
)

In [ ]:
def convert_to_gdf(df, lon_col='longitude', lat_col='latitude', crs='EPSG:4326'):
    """Convert a DataFrame with longitude and latitude columns to a GeoDataFrame."""
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    return gdf

In [ ]:
hydrant_complaints = convert_to_gdf(hydrant_complaints)
ac_complaints = convert_to_gdf(ac_complaints)
ventilation_complaints = convert_to_gdf(ventilation_complaints)
power_complaints = convert_to_gdf(power_complaints)

In [ ]:
zip_code_gdf = gpd.read_file('https://github.com/fedhere/PUI2015_EC/raw/refs/heads/master/mam1612_EC/nyc-zip-code-tabulation-areas-polygons.geojson')

In [ ]:
heat_zipcode_counts = heat_incidents.groupby('zipcode').size()
heat_zipcode_counts.name = 'heat_incident_counts'

In [ ]:
ax = zip_code_gdf.set_index('postalCode').join(heat_zipcode_counts).plot('heat_incident_counts')
hydrant_complaints.plot(ax =ax, color='red', markersize=1)
# power_complaints.plot(ax =ax, color='red', markersize=1)